# Analyse des conventions du merge CAN/NHR
Sources du 2 octobre 2026. Analyse seule : aucune modification des CSV sources ni du merge.
Exécuter depuis la racine de can-py-main. Les contrôles portent sur cet enregistrement de décharge; la charge reste à vérifier sur un autre essai.
Les SoP sont des limites, les consignes sont des paramètres configurés, et les mesures sont des observations.


In [1]:
import csv, json, hashlib, statistics, collections
from pathlib import Path
from datetime import datetime
root = Path("C:/nhr-rt-runs/ModuleTests/can")
can_path = root / "can_capture_20261002_115451.csv"
merged_path = root / "merged_capture_20261002_115451.csv"
report = json.loads((root / "merged_capture_20261002_115451.report.json").read_text())
manifest = json.loads((root / "can_capture_20261002_115451.manifest.json").read_text())
def read(path):
    with path.open(encoding="utf-8-sig", newline="") as f:
        return list(csv.DictReader(f))
merged = read(merged_path)
can = read(can_path)
nhr_path = Path(report["nhr_artifact"]["path"])
nhr = read(nhr_path)
assert hashlib.sha256(nhr_path.read_bytes()).hexdigest() == report["nhr_artifact"]["sha256"]
dbc = Path("dbc/6.50.6.0_BMSM_module_test.dbc")
assert hashlib.sha256(dbc.read_bytes()).hexdigest() == manifest["dbc"]["sha256"]
assert len(merged) == len(nhr) == 633
assert all(all(m["nhr_" + k] == v for k, v in n.items()) for m, n in zip(merged, nhr))
signals = report["signals"]
state = {}; index = 0; mismatches = 0
for m in merged:
    stamp = datetime.fromisoformat(m["nhr_timestamp_utc"])
    while index < len(can) and datetime.fromisoformat(can[index]["timestamp_utc"]) <= stamp:
        raw = can[index]
        for s in signals:
            if raw.get(s, "").strip(): state[s] = raw[s]
        index += 1
    for s in signals:
        expected = state.get(s, "")
        if s == "batteryCurrent" and expected: expected = str(-float(expected))
        mismatches += m["can_" + s] != expected
assert mismatches == 0
print("CAN rows:", len(can), "Merged rows:", len(merged), "UTC duration s:", report["overlap_duration_s"])
print("Verified: DBC hash, NHR artifact hash, unchanged NHR fields, all selected CAN values against backward as-of join.")
discharge = [r for r in merged if r["nhr_state"] == "discharge"]
for k in ["nhr_current_a", "can_batteryCurrent", "nhr_power_w", "nhr_setpoint_current_a", "nhr_setpoint_power_w", "can_maxChargePower", "can_maxDischargePower"]:
    print(k, "discharge median:", statistics.median(float(r[k]) for r in discharge))
active = [r for r in merged if abs(float(r["nhr_current_a"])) > 10]
print("Active current signs match:", sum(float(r["nhr_current_a"])*float(r["can_batteryCurrent"]) > 0 for r in active), "/", len(active))
print("Median BMS minus NHR voltage, discharge V:", statistics.median(float(r["can_batteryVoltage"])-float(r["nhr_voltage_v"]) for r in discharge))
print("Median implied voltage-drop/current ratio, discharge ohm:", statistics.median((float(r["can_batteryVoltage"])-float(r["nhr_voltage_v"]))/abs(float(r["nhr_current_a"])) for r in discharge))
print("NHR temperature range:", min(float(r["nhr_temperature_c"]) for r in merged), max(float(r["nhr_temperature_c"]) for r in merged))
print("BMS maximum temperature range:", min(float(r["can_maxCellTemp"]) for r in merged), max(float(r["can_maxCellTemp"]) for r in merged))
print("OFF rows retaining nonzero current/power setpoints:", sum(r["nhr_state"] == "off" and float(r["nhr_setpoint_current_a"]) != 0 and float(r["nhr_setpoint_power_w"]) != 0 for r in merged))
print("Missing throughout:", [s for s in signals if all(r["can_"+s+"_status"] == "missing" for r in merged)])
print("Power status:", dict(collections.Counter(r["can_maxDischargePower_status"] for r in merged)))
print("State counts:", dict(collections.Counter(r["nhr_state"] for r in merged)))
print("Sample 100:", {k:merged[100][k] for k in ["nhr_timestamp_utc", "nhr_state", "nhr_current_a", "nhr_power_w", "can_maxChargePower", "can_maxDischargePower", "nhr_setpoint_current_a", "nhr_setpoint_power_w"]})


CAN rows: 94914 Merged rows: 633 UTC duration s: 63.33320999145508
Verified: DBC hash, NHR artifact hash, unchanged NHR fields, all selected CAN values against backward as-of join.
nhr_current_a discharge median: -192.00965881347656
can_batteryCurrent discharge median: -192.6435546875
nhr_power_w discharge median: -14497.1875
nhr_setpoint_current_a discharge median: 192.0
nhr_setpoint_power_w discharge median: 20000.0
can_maxChargePower discharge median: -15.5
can_maxDischargePower discharge median: 43.0625
Active current signs match: 273 / 273
Median BMS minus NHR voltage, discharge V: 1.8895645141601562
Median implied voltage-drop/current ratio, discharge ohm: 0.009840977033495029
NHR temperature range: 157.19703674316406 157.30621337890625
BMS maximum temperature range: 37.234375 39.15625
OFF rows retaining nonzero current/power setpoints: 356
Missing throughout: ['CELL_T_1_1', 'CELL_T_3_1', 'CELL_V_0_0', 'CELL_V_0_1', 'CELL_V_0_2', 'CELL_V_0_3', 'CELL_V_0_4', 'CELL_V_0_5', 'CELL_V_